# Advanced single-trial fNIRS finger tapping classification

This notebook analyzes a multi-subject finger-tapping dataset and trains a
Linear Discriminant Analysis (LDA) classifier to distinguish single trials of
left-hand tapping from a resting/control condition.

Compared to the introductory
[50_finger_tapping_lda_classification.ipynb](./50_finger_tapping_lda_classification.ipynb)
notebook, this one goes into more depth:

- Channels are pruned per subject using the Scalp Coupling Index (SCI) and
  Peak Spectral Power (PSP) quality metrics.
- A GLM-based short-separation channel (SSC) regression is used to remove the
  systemic/superficial component from the long-channel concentration signal,
  and classification accuracy with and without this regression is compared
  via cross-validation.
- At the end of the notebook, the same trials are additionally analyzed in
  **image (parcel) space**: channel-space optical density is projected onto
  the cortical surface via diffuse optical tomography (DOT) image
  reconstruction, aggregated into brain parcels, and classified with the same
  features and classifier used for channel space — allowing a direct
  comparison of classification performance between channel-space and
  parcel-space representations.

On a basic level, this notebook illustrates some of the elements investigated
more rigorously in <cite data-cite="Fischer2026">(Fischer et al., 2026)</cite>,
who show that single-trial fNIRS decoding accuracy improves systematically
with higher optode density, model-based (GLM) noise regression, and image
reconstruction. This notebook uses a single, comparatively sparse probe and a
simple LDA classifier, so it should be read as a minimal illustration of the
*type* of comparison made in that paper (channel space vs. image/parcel
space, with/without regression), not as a reproduction of its results.

**PLEASE NOTE:** For simplicity's sake we still skip several preprocessing
steps that a rigorous analysis would include (e.g. motion artifact rejection
beyond TDDR/wavelet correction, physiological noise regression beyond
short-separation channels). These are the subject of other example notebooks.
The purpose of this notebook is to demonstrate how cedalion interfaces with
scikit-learn for both channel-space and image-space classification.

In [ ]:
# This cells setups the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

In [ ]:
import matplotlib.pyplot as p
import numpy as np
import pyvista as pv
import xarray as xr
from collections import defaultdict

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import auc, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

import cedalion
import cedalion.nirs
from cedalion import units
from cedalion.data import get_multisubject_fingertapping_snirf_paths
from cedalion.sigproc.frequency import freq_filter
import cedalion.data
import cedalion.dot
import cedalion.io
import cedalion.sigproc.quality as quality
import cedalion.sigproc.motion as motion
import cedalion.vis.anatomy
import cedalion.vis.anatomy.sensitivity_matrix
import cedalion.vis.blocks as vbx
import cedalion.geometry.landmarks
import cedalion.models.glm as glm
import cedalion.mlutils as mlutils

# set to True for interactive (rotatable) 3D plots, False for static images
INTERACTIVE_PLOTS = False
pv.set_jupyter_backend("server" if INTERACTIVE_PLOTS else "static")

xr.set_options(display_max_rows=3, display_values_threshold=50)
np.set_printoptions(precision=4)

## Loading raw CW-NIRS data from SNIRF files

This notebook uses a finger-tapping dataset in BIDS layout provided by [Rob Luke](https://github.com/rob-luke/BIDS-NIRS-Tapping). It can can be downloaded via `cedalion.data`.

Cedalion's `read_snirf` method returns a list of `Recording` objects. These are containers for timeseries and adjunct data objects.

In [ ]:
fnames = get_multisubject_fingertapping_snirf_paths()
subjects = [f"sub-{i:02d}" for i in [1, 2, 3, 4, 5]]

sci_threshold = 0.6
psp_threshold = 0.05
window_length = 5 * units.s
pc_clean_threshold = 0.5

# store data of different subjects in a dictionary
data = {}
for subject, fname in zip(subjects, fnames):
    records = cedalion.io.read_snirf(fname)
    rec = records[0]

    # Cedalion registers an accessor (attribute .cd ) on pandas DataFrames.
    # Use this to rename trial_types inplace.
    rec.stim.cd.rename_events(
        {
            "1.0": "control",
            "2.0": "Tapping/Left",
            "3.0": "Tapping/Right",
            "15.0": "sentinel",
        }
    )

    # remove unused events, sort by onset
    rec.stim = (
        rec.stim[rec.stim.trial_type.isin(["control", "Tapping/Left"])]
        .sort_values("onset")
        .reset_index(drop=True)
    )

    rec.geo3d = cedalion.geometry.landmarks.normalize_landmarks_labels(rec.geo3d)

    rec["od"] = cedalion.nirs.cw.int2od(rec["amp"])
    rec["od_tddr"] = motion.tddr(rec["od"])
    rec["od_wavelet"] = motion.wavelet(rec["od_tddr"])

    # SCI & PSP
    sci, sci_mask = quality.sci(rec["od_wavelet"], window_length, sci_threshold)
    psp, psp_mask = quality.psp(rec["od_wavelet"], window_length, psp_threshold)
    sci_psp_mask = sci_mask & psp_mask
    
    rec.aux_obj["perc_time_clean"] = sci_psp_mask.sum(dim="time") / len(sci_psp_mask.time)

    rec.masks["clean_mask"] = rec.aux_obj["perc_time_clean"] > pc_clean_threshold
    ncleanchannels = rec.masks["clean_mask"].sum().values

    print(
        f"{subject}: number of clean channels: "
        f"{ncleanchannels}/{rec['amp'].sizes['channel']}"
    )

    rec["od_pruned"] = rec["od_wavelet"].sel(channel=rec.masks["clean_mask"])
    
    dpf = xr.DataArray(
        [6, 6],
        dims="wavelength",
        coords={"wavelength": rec["amp"].wavelength},
    )

    rec["conc"] = cedalion.nirs.cw.od2conc(rec["od_pruned"], rec.geo3d, dpf)

    rec["conc_freqfilt"] = freq_filter(
        rec["conc"], fmin=0.01 * units.Hz, fmax=0.5 * units.Hz
    )

    data[subject] = rec

Inspect Trials

In [ ]:
display(
    data["sub-01"]
    .stim.groupby("trial_type")[["trial_type"]]
    .count()
    .rename({"trial_type": "# trials"}, axis=1) # rename column
)


Inspect Montage

In [ ]:
cedalion.vis.anatomy.plot_montage3D(data["sub-01"]["amp"], data["sub-01"].geo3d, landmarks=["Nz", "Iz", "LPA", "RPA", "Cz"])

### Plot preprocessed data
Illustrate for a single subject and channel the effect of the preprocessing.

In [ ]:
for i_sub, subject in enumerate(subjects):

    rec = data[subject]
    channel = "S2D4"
    od_ylims = (-.1,.1)
    conc_ylims = (-1.5,1.5)

    f, ax = p.subplots(4, 1, figsize=(9, 6), sharex=True)


    ax[0].plot(rec["od"].time, rec["od"].sel(channel=channel, wavelength=760), c="#4daf4a", label="760nm")
    ax[0].plot(rec["od"].time, rec["od"].sel(channel=channel, wavelength=850), c="#984ea3", label="850nm")
    ax[0].set_ylabel("OD")
    ax[0].set_title("OD")
    ax[0].set_ylim(*od_ylims)

    ax[1].plot(rec["od_wavelet"].time, rec["od_wavelet"].sel(channel=channel, wavelength=760), c="#4daf4a", label="760nm")
    ax[1].plot(rec["od_wavelet"].time, rec["od_wavelet"].sel(channel=channel, wavelength=850), c="#984ea3", label="850nm")
    ax[1].set_ylabel("OD")
    ax[1].set_title("OD after TDDR and wavelet correction")
    ax[0].set_ylim(*od_ylims)

    ax[2].plot(rec["conc"].time, rec["conc"].sel(channel=channel, chromo="HbO"), "r-", label="HbO")
    ax[2].plot(rec["conc"].time, rec["conc"].sel(channel=channel, chromo="HbR"), "b-", label="HbR")
    ax[2].set_ylabel("$\Delta c$ / $\mu M$")
    ax[2].set_title("hemoglobin concentrations")
    ax[2].set_ylim(*conc_ylims)

    ax[3].plot(rec["conc_freqfilt"].time, rec["conc_freqfilt"].sel(channel=channel, chromo="HbO"), "r-", label="HbO")
    ax[3].plot(rec["conc_freqfilt"].time, rec["conc_freqfilt"].sel(channel=channel, chromo="HbR"), "b-", label="HbR")
    ax[3].set_ylabel("$\Delta c$ / $\mu M$")
    ax[3].set_title("hemoglobin concentrations after freq. filter")
    ax[3].set_ylim(*conc_ylims)

    ax[0].set_xlim(1500, 1700)
    ax[3].set_xlabel("time / s")

    for a in ax:
        cedalion.vis.blocks.plot_stim_markers(a, rec.stim, y=1.)
        a.legend(loc="upper left", ncol=5, fontsize=8)
    f.tight_layout()

## Show channel quality per subject

In [ ]:
f,ax = p.subplots(1,5, figsize=(15,3))
for i, sub in enumerate(subjects):
    rec = data[sub]

    """
    od_var = quality.measurement_variance(rec["od_wavelet"], calc_covariance=False)
    w = (1 / od_var)
    w /= w.max("channel")

    
    cedalion.vis.anatomy.scalp_plot(
        rec["od"],
        rec.geo3d,
        #/od_var.isel(wavelength=0),
        w.isel(wavelength=0),
        ax=ax[0,i],
        optode_size=2,
        title=sub,
        cmap="RdYlGn",
        vmin=0,
        vmax=0.1,
        #cb_label="meas. var."
    )
    """
    cedalion.vis.anatomy.scalp_plot(
        rec["od"],
        rec.geo3d,
        rec.aux_obj["perc_time_clean"],
        ax=ax[i],
        optode_size=2,
        title=sub,
        cmap="RdYlGn",
        vmin=0.0,
        vmax=1,
        #cb_label="fraction of clean time"
    )
    
f.suptitle("Percentage of clean time per channel")
f.tight_layout()


## Short-channel regression and epoching

Define a function that, for a single subject:
- splits the trials into 4 cross-validation folds
- defines a GLM design matrix with HRF and short-channel regressors
- blanks the design matrix during the test trials of each cv. fold
- fits the GLM and subtracts the component explained by the short-channel regressor
- splits the SSC-corrected time series into epochs
- subtracts a baseline from each epoch
- returns a list of arrays, one per cv fold; each array contains all epochs with
  coordinates attached that allow train and test trials to be distinguished

In [ ]:
def process_single_subject(rec, subtract_global_component : bool) -> list[xr.DataArray]:

    n_splits = 4
    before = 2 * cedalion.units.s
    after = 15 * cedalion.units.s

    cv_folds = []

    # separate long and short channels
    rec["conc_long"], rec["conc_short"] = cedalion.nirs.split_long_short_channels(
        rec["conc_freqfilt"], rec.geo3d, distance_threshold=15.0 * units.mm
    )

    # define the design matrix
    dms = (
        glm.design_matrix.hrf_regressors(
            rec["conc_long"],
            rec.stim,
            glm.Gamma(tau=0 * units.s, sigma=3 * units.s),
        )
        & glm.design_matrix.average_short_channel_regressor(rec["conc_short"])
    )


    # split trials into train and test for n_splits CV folds
    for i_split, (df_stim_train, df_stim_test) in enumerate(
        mlutils.cv.create_cv_splits(rec.stim, n_splits)
    ):

        if subtract_global_component:
            # zero-out design matrix in test segment
            dms_masked = mlutils.cv.mask_design_matrix(
                dms,
                df_stim_test,
                before=before,
                after=after,
            )

            # fit long channels with masked design matrix
            result = glm.fit(rec["conc_long"], dms_masked, noise_model="ols")

            # compute component explained by short-channel regressor
            short_component = glm.predict(
                rec["conc_long"],
                result.sm.params.sel(regressor=["short"]),
                dms,
            )
            short_component = short_component.pint.quantify(rec["conc_long"].pint.units)

            # subtract short component
            conc_for_epoching = rec["conc_long"] - short_component
        else:
            conc_for_epoching = rec["conc_long"]

        # split time series into epochs
        epochs = conc_for_epoching.cd.to_epochs(
            rec.stim,
            #["Tapping/Left", "Tapping/Right"],
            ["control", "Tapping/Left"],
            before=before,
            after=after,
        )

        # baseline correction
        baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")
        epochs = epochs - baseline

        # assign train-/test-set membership...
        is_train = np.zeros(epochs.sizes["epoch"], dtype=bool)
        is_test = np.zeros(epochs.sizes["epoch"], dtype=bool)
        is_train[df_stim_train.index.values] = True
        is_test[df_stim_test.index.values] = True

        # ... and digitized trial labels ... 
        # (0,1,.. instead of "Tapping/Left", "Tapping/Right"...)
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(epochs.trial_type.values)

        # ... as coordinates to the DataArray
        epochs = epochs.assign_coords(
            {
                "is_train": ("epoch", is_train),
                "is_test": ("epoch", is_test),
                "y": ("epoch", y),
            }
        )

        cv_folds.append(epochs)
    
    return cv_folds

In [ ]:
cv_folds = process_single_subject(data["sub-01"], subtract_global_component=False)

display(cv_folds[0])
display(cv_folds[-1])

In [ ]:
for sub in subjects:

    f = p.figure(figsize=(16, 8), constrained_layout=True)
    subfigs = f.subfigures(1, 2)

    for subfig, subtract_global_component in zip(subfigs, [True, False]):

        cv_folds = cv_folds = process_single_subject(data[sub], subtract_global_component)

        blockaverage = cv_folds[0].groupby("trial_type").mean("epoch")

        # Plot block averages. Please ignore errors if the plot is too small in the HD case

        noPlts2 = int(np.ceil(np.sqrt(len(blockaverage.channel))))

        ax = subfig.subplots(noPlts2, noPlts2).flatten()

        for i_ch, ch in enumerate(blockaverage.channel):
            for ls, trial_type in zip(["-", "--"], blockaverage.trial_type):
                ax[i_ch].plot(blockaverage.reltime, blockaverage.sel(chromo="HbO", trial_type=trial_type, channel=ch), "r", lw=2, ls=ls)
                ax[i_ch].plot(blockaverage.reltime, blockaverage.sel(chromo="HbR", trial_type=trial_type, channel=ch), "b", lw=2, ls=ls)

            ax[i_ch].grid(1)
            ax[i_ch].set_title(ch.values)
            ax[i_ch].set_ylim(-.3, .5)
            ax[i_ch].set_axis_off()
            ax[i_ch].axhline(0, c="k")
            ax[i_ch].axvline(0, c="k")

        for i in range(len(blockaverage.channel), len(ax)):
            ax[i].set_axis_off()

        subfig.suptitle(f"sub. glob. comp: {subtract_global_component}")
        #subfig.suptitle(f"HbO: r | HbR: b | left: - | right: -- | sub. glob. comp: {subtract_global_component}")

    f.suptitle(f"{sub} | HbO: r | HbR: b | {blockaverage.trial_type.values[0]}: - | {blockaverage.trial_type.values[1]}: --")

The function `mlutils.features.epoch_features` calculates common features of the hemodynamic response, such as slope, mean, maximum, minimum and area under the curve. For each feature type, a time range can be specified over which the feature is calculated. In the present case, this yields features for each channel and chromophore, which are then stacked into a single feature dimension. The resulting array has the shape expected by scikit-learn, ($N_{samples}$, $N_{features}$).

In [ ]:
X = mlutils.features.epoch_features(
    cv_folds[0],
    feature_types=["slope", "mean", "max", "min", "auc"],
    reltime_slices={
        "slope": slice(0, 9),
        "mean": slice(3, 10),
        "max": slice(2, 8),
        "min": slice(2, 8),
    },
)
X

In [ ]:
accuracies = defaultdict(list)


for sub in subjects:
    for subtract_global_component in [True, False]:
        key = (sub, subtract_global_component)

        rec = data[sub]
        cv_folds = process_single_subject(rec, subtract_global_component)

        for epochs in cv_folds:
            # extract features
            X = mlutils.features.epoch_features(
                epochs.sel(chromo="HbO"),  # HbO only
                feature_types=["slope", "max", "mean"],
                reltime_slices={
                    "slope": slice(0, 9),
                    # "auc" : slice(0, 9),
                    "mean": slice(3, 10),
                    "max": slice(2, 8),
                },
            )

            # separate train and test sets
            X_train = X[X.is_train]
            y_train = X_train.y
            X_test = X[X.is_test]
            y_test = X_test.y

            # train a LDA classifier

            clf = make_pipeline(
                StandardScaler(),
                LinearDiscriminantAnalysis(
                    n_components=1, solver="lsqr", shrinkage="auto"
                ),
            )
            # clf = LinearDiscriminantAnalysis(n_components=1, solver='lsqr', shrinkage="auto")
            clf.fit(X_train, y_train)

            # evaluate perfomance
            accuracy = clf.score(X_test, y_test)

            accuracies[key].append(accuracy)
            print(
                f"{sub} - #train: {len(y_train)} #test: {len(y_test)} #features: {X_train.shape[1]} accuracy: {accuracy:.3f}"
            )

        print(
            rf"{sub} subtract global: {subtract_global_component} - average accuracy over cross-validation splits: {np.mean(accuracies[key]):.3f} ± {np.std(accuracies[key]):.3f}"
        )
        print("-" * 80)


In [ ]:
from statsmodels.stats.proportion import proportion_confint

def acc_and_ci(fold_accs, n_per_fold=15):
    """Pooled accuracy and Wilson-CI error-bar lengths (lower, upper)."""
    n_total = n_per_fold * len(fold_accs)
    n_correct = round(sum(fold_accs) * n_per_fold)
    mean = n_correct / n_total
    lo, hi = proportion_confint(n_correct, n_total, alpha=0.05, method="wilson")
    return mean, mean - lo, hi - mean

f, ax = p.subplots(figsize=(8, 6))
x = np.arange(len(subjects))

for dx, use_scc, fmt, label in [(-.1, False, "rs", "No SS correction"),
                                (+.1, True,  "gs", "SS correction")]:
    mean, lo, hi = np.array([acc_and_ci(accuracies[s, use_scc]) for s in subjects]).T
    ax.errorbar(x + dx, mean, yerr=[lo, hi], fmt=fmt, label=label)

ax.set_xticks(x)
ax.set_xticklabels(subjects)
ax.axhline(0.5, c="k", ls="--", label="chance")
ax.legend(loc="center right")
ax.set_ylim(0.45, 1.05)
ax.set_ylabel("accuracy")

## Bonus: classification in image (parcel) space

So far, all classification was done on **channel-space** concentration data,
where each feature is tied to a specific source-detector pair. Cedalion can
also solve the DOT inverse problem to project channel-space data onto the
cortical surface ("image reconstruction"), which yields a representation that
is independent of the particular probe geometry and can be interpreted
anatomically.

**Caveat:** the probe used in this dataset is a comparatively sparse, single-
distance montage covering only motor cortex — far from the high-density
layouts for which DOT image reconstruction is designed and validated. Image
reconstruction from such a sparse probe is inherently ill-posed and cannot
recover the same spatial detail or SNR that a dense array would provide.
We reconstruct images here anyway, not because it is the ideal use case, but
to demonstrate *how* to go from channel-space data to parcel-space features
and classify on them with cedalion — the same workflow applies directly to
higher-density probes, where image-space classification is expected to be
more advantageous (see the note on <cite data-cite="Fischer2026">(Fischer et
al., 2026)</cite> above).

In this section we:

1. Load a **precomputed sensitivity (Adot) matrix** for this probe montage
   (`cedalion.data.get_precomputed_sensitivity`). This is the linear forward
   operator that maps absorption changes at every vertex of the cortical
   surface to optical density changes at every channel/wavelength.
2. Build an `cedalion.dot.ImageRecon` object that solves the (regularized)
   inverse problem, and use it to reconstruct HbO/HbR concentration images
   directly from optical density.
3. Determine which cortical **parcels** (Schaefer2018 atlas) this probe is
   actually sensitive to, using `ForwardModel.parcel_sensitivity`, and
   restrict the analysis to those parcels only — regions the montage cannot
   see should not be used as classifier features.
4. Reconstruct images from `od_wavelet` — the optical density time series
   right after TDDR and wavelet motion correction (the same starting point
   used for the channel-space `SS correction: False` condition above), i.e.
   **without** any short-channel/GLM regression — and average vertex-space
   images into brain parcels.
5. Extract the same slope/mean/max features used for channel-space
   classification, this time per parcel instead of per channel, and train
   and cross-validate the same LDA pipeline.
6. Compare parcel-space accuracy to the channel-space accuracy obtained
   above (`No SS correction` condition), since both use the same
   preprocessing stage and no short-channel regression.

In [ ]:
HEAD_MODEL = "icbm152"

# precomputed sensitivity (Adot) matrix for this probe montage, on the
# standard icbm152 head model. dims: (channel, vertex, wavelength);
# vertex coords include "is_brain" and "parcel" (Schaefer2018 atlas label).
Adot = cedalion.data.get_precomputed_sensitivity("fingertapping", HEAD_MODEL)

# the head model itself, used below for plotting the sensitivity profile
# and the sensitive parcels on the brain/scalp surfaces.
head = cedalion.dot.get_standard_headmodel(HEAD_MODEL)

# image reconstruction operator: OD -> HbO/HbR concentration images.
# alpha_spatial=None -> plain Tikhonov (measurement-side) regularization only.
recon = cedalion.dot.ImageRecon(
    Adot,
    recon_mode="mua2conc",
    brain_only=True,
    alpha_meas=0.01,
    alpha_spatial=None,
    apply_c_meas=False,
    spatial_basis_functions=None,
)

# which parcels can this probe actually see? A parcel is considered
# "sensitive" if a plausible HbO/HbR change within it would produce an
# observable dOD in at least one channel/wavelength.
parcel_dOD, parcel_mask = cedalion.dot.ForwardModel.parcel_sensitivity(
    Adot, chan_droplist=None, dOD_thresh=0.001, minCh=1, dHbO=10, dHbR=-3
)
sensitive_parcels = parcel_mask.where(parcel_mask, drop=True)["parcel"].values.tolist()

print(f"probe is sensitive to {len(sensitive_parcels)} of {parcel_mask.sizes['parcel']} parcels")

### Sensitivity profile

`Adot` describes, for every vertex on the cortical surface, how strongly a
local absorption change affects the measured optical density (summed over
channels). Plotting this on the head model shows where the probe has
useful sensitivity — here, unsurprisingly, over the sensorimotor cortex.

In [ ]:
sens_plotter = cedalion.vis.anatomy.sensitivity_matrix.Main(
    sensitivity=Adot,
    brain_surface=head.brain,
    head_surface=head.scalp,
)
sens_plotter.plot(high_th=0, low_th=-3)
sens_plotter.plt.show()

### Sensitive parcels used for classification

Each colored region below is one of the `sensitive_parcels` retained for the
parcel-space classification below (colored by its Schaefer2018 atlas color);
parcels the probe cannot reliably observe are left gray and excluded from
the feature set.

In [ ]:
headmodel_files = cedalion.data.get_icbm152_headmodel_files()
parcel_colors = cedalion.io.read_parcel_colors(
    headmodel_files.basedir / headmodel_files.parcel_colors
)

# color sensitive parcels by their atlas color, gray out the rest
color = [
    parcel_colors.get(parcel, [0, 0, 0]) if parcel in sensitive_parcels else [0.85, 0.85, 0.85]
    for parcel in head.brain.vertices.parcel.values
]

plt = pv.Plotter()
vbx.plot_surface(plt, head.brain, color=color, silhouette=True)
vbx.camera_at_cog(plt, head.brain, rpos=[400, 0, 400], fit_scene=True)
plt.show()

For each subject, epoch `od_wavelet` (no short-channel regression), reconstruct
an image for every single epoch at once (the `epoch` and `reltime` dimensions
simply pass through `ImageRecon.reconstruct`), average the brain vertices into
parcels, and keep only the `sensitive_parcels` determined above. The same
`mlutils.cv.create_cv_splits` train/test split used for channel-space
classification is reused here, so that both analyses are evaluated on
identical trials.

In [ ]:
before = 2 * units.s
after = 15 * units.s
n_splits = 4

accuracies_parcel = defaultdict(list)

for sub in subjects:
    rec = data[sub]

    # epoch the OD time series right after TDDR + wavelet correction
    # (no short-channel regression)
    epochs_od = rec["od_wavelet"].cd.to_epochs(
        rec.stim, ["control", "Tapping/Left"], before=before, after=after
    )
    baseline = epochs_od.sel(reltime=(epochs_od.reltime < 0)).mean("reltime")
    epochs_od = epochs_od - baseline

    # reconstruct all epochs at once: (chromo, vertex, epoch, reltime)
    img = recon.reconstruct(epochs_od)

    # average brain vertices into parcels, keep only parcels the probe can see
    img_parcel = img.groupby("parcel").mean()
    img_parcel = img_parcel.sel(parcel=img_parcel.parcel.isin(sensitive_parcels))

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(epochs_od.trial_type.values)
    img_parcel = img_parcel.assign_coords(y=("epoch", y))

    for df_stim_train, df_stim_test in mlutils.cv.create_cv_splits(rec.stim, n_splits):
        # same feature types/windows as used for channel-space classification
        X = mlutils.features.epoch_features(
            img_parcel.sel(chromo="HbO"),  # HbO only
            feature_types=["slope", "max", "mean"],
            reltime_slices={
                "slope": slice(0, 9),
                "mean": slice(3, 10),
                "max": slice(2, 8),
            },
        )

        X_train = X.isel(epoch=df_stim_train.index.values)
        y_train = X_train.y
        X_test = X.isel(epoch=df_stim_test.index.values)
        y_test = X_test.y

        # same LDA pipeline as used for channel-space classification
        clf = make_pipeline(
            StandardScaler(),
            LinearDiscriminantAnalysis(n_components=1, solver="lsqr", shrinkage="auto"),
        )
        clf.fit(X_train, y_train)

        accuracy = clf.score(X_test, y_test)
        accuracies_parcel[sub].append(accuracy)
        print(
            f"{sub} - #train: {len(y_train)} #test: {len(y_test)} "
            f"#features: {X_train.shape[1]} accuracy: {accuracy:.3f}"
        )

    print(
        f"{sub} parcel-space - average accuracy over cross-validation splits: "
        f"{np.mean(accuracies_parcel[sub]):.3f} ± {np.std(accuracies_parcel[sub]):.3f}"
    )
    print("-" * 80)

Compare per-subject accuracy between channel-space (without short-channel
regression) and parcel-space classification. Both use identical
train/test splits, feature types, feature windows, and classifier, so any
difference in accuracy reflects the change in data representation only.

In [ ]:
f, ax = p.subplots(figsize=(8, 6))
x = np.arange(len(subjects))

mean_ch, lo_ch, hi_ch = np.array([acc_and_ci(accuracies[s, False]) for s in subjects]).T
mean_pa, lo_pa, hi_pa = np.array([acc_and_ci(accuracies_parcel[s]) for s in subjects]).T

ax.errorbar(x - .1, mean_ch, yerr=[lo_ch, hi_ch], fmt="rs", label="Channel space (No SS correction)")
ax.errorbar(x + .1, mean_pa, yerr=[lo_pa, hi_pa], fmt="bo", label="Parcel space (image recon.)")

ax.set_xticks(x)
ax.set_xticklabels(subjects)
ax.axhline(0.5, c="k", ls="--", label="chance")
ax.legend(loc="center right")
ax.set_ylim(0.45, 1.05)
ax.set_ylabel("accuracy")
ax.set_title("Channel-space vs. parcel-space classification accuracy")

## References

In [ ]:
cedalion.bib.dump_to_notebook()